**The required libraries are imported here**

In [13]:
from five_safes_tes_workbench.workbench import Workbench

In [14]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parents[1] / "contingency-tables"))

from contingency_table_utils import ContingencyTable, aggregate_tables

**Validate workbench configuration**

In [ ]:
wb = Workbench()

wb.validate(config_path="config.yaml") #type: ignore

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Validation successful
INFO | Config: project='DelphiDemo' tes_base_url='https://api.5s-tes.federated-research.com/' minio_sts_endpoint='https://api.minio.5s-tes.federated-research.com/sts' minio_endpoint='https://api.minio.5s-tes.federated-research.com/' minio_output_bucket='126104output' tres=['Nottingham TRE 01', 'Nottingham TRE 02']
INFO | Auth mode: AuthMode.ACCESS_TOKEN


**Define SQL query**

The query checks the `person` table and the `condition_occurrence` table to classify each person by:

- whether they have primary malignant neoplasm of skin
- whether they have hypertension

The query then groups by these two categorical variables and returns the count for each combination. 

In [4]:
hypertension_neoplasm_query = """
WITH hypertension AS (
  SELECT
    person_id,
    CASE
      WHEN person_id IN (
        SELECT person_id
        FROM "DelphiDemo".condition_occurrence
        WHERE condition_concept_id = 320128
      ) THEN 'has_hypertension' ELSE 'no_hypertension' END AS hypertension_status
  FROM "DelphiDemo".person
)

SELECT
  CASE
    WHEN p.person_id IN (
      SELECT person_id
      FROM "DelphiDemo".condition_occurrence
      WHERE condition_concept_id = 139750
    ) THEN 'with'
    ELSE 'without'
    END AS neoplasm_status,
  hypertension.hypertension_status,
  COUNT(p.person_id) as n
FROM "DelphiDemo".person p
  JOIN hypertension ON p.person_id = hypertension.person_id
GROUP BY neoplasm_status, hypertension_status
"""


wb.build_tes.simple_sql(
    name="hypertension x neoplasm contingency table",
    query=hypertension_neoplasm_query
)

wb.submit()

INFO | Building TES task from template: 'simple_sql'
INFO | Resolving template: 'simple_sql'
INFO | TES Task built successfully
INFO | TES payload:
{
   "name": "hypertension x neoplasm contingency table",
   "description": "Simple SQL Task",
   "outputs": [
      {
         "url": "s3://",
         "path": "/outputs",
         "type": "DIRECTORY",
         "name": "Output",
         "description": "Output results"
      }
   ],
   "executors": [
      {
         "image": "harbor.federated-analytics.ac.uk/5s-tes-analysis-tools/5s-tes-analysis-tools-tre-sqlpg:1.0.0",
         "command": [
            "--Output=/outputs/output.csv",
            "--Query=\nWITH hypertension AS (\n  SELECT\n    person_id,\n    CASE\n      WHEN person_id IN (\n        SELECT person_id\n        FROM \"DelphiDemo\".condition_occurrence\n        WHERE condition_concept_id = 320128\n      ) THEN 'has_hypertension' ELSE 'no_hypertension' END AS hypertension_status\n  FROM \"DelphiDemo\".person\n)\n\nSELECT\n  CA

'1195'

**Fetch approved outputs**

- After the task has completed and the outputs have been approved for egress.

In [5]:
wb.fetch_outputs()

INFO | Using provided access token
INFO | Exchanging bearer token for MinIO credentials via STS (https://api.minio.5s-tes.federated-research.com/sts)
INFO | MinIO client initialised (endpoint=https://api.minio.5s-tes.federated-research.com/, secure=True)
INFO | Child task info: 1196, TRE: Nottingham TRE 01, status: Completed
INFO | Found 1 result object(s) for task 1196
INFO | Downloading result object: 1196/output.csv
INFO | Downloaded 1196/output.csv -> /Users/karthik/Developer/Health-Informatics/5s-TES-notebooks/workbench-delphi/notebook-analysis/output/Nottingham TRE 01/1196/output.csv
INFO | Child task info: 1197, TRE: Nottingham TRE 02, status: Completed
INFO | Found 1 result object(s) for task 1197
INFO | Downloading result object: 1197/output.csv
INFO | Downloaded 1197/output.csv -> /Users/karthik/Developer/Health-Informatics/5s-TES-notebooks/workbench-delphi/notebook-analysis/output/Nottingham TRE 02/1197/output.csv


{'Nottingham TRE 01': [PosixPath('/Users/karthik/Developer/Health-Informatics/5s-TES-notebooks/workbench-delphi/notebook-analysis/output/Nottingham TRE 01/1196/output.csv')],
 'Nottingham TRE 02': [PosixPath('/Users/karthik/Developer/Health-Informatics/5s-TES-notebooks/workbench-delphi/notebook-analysis/output/Nottingham TRE 02/1197/output.csv')]}

**Define output paths**

- These paths point to the approved CSV outputs from Nottingham TRE 01 and Nottingham TRE 02.

- The task IDs may change after each submission, so update these paths after fetching new outputs.

In [7]:
contingency_paths = [
    "./output/Nottingham TRE 01/1196/output.csv",
    "./output/Nottingham TRE 02/1197/output.csv"
]

pre_contingency_tables = [ContingencyTable(pd.read_csv(path)) for path in contingency_paths]

In [ ]:
# Aggregate TRE-level contingency tables into a single contingency table

aggregated = aggregate_tables(pre_contingency_tables)
contingency_table = aggregated.contingency_table["n"]

display(contingency_table)

hypertension_status,has_hypertension,no_hypertension
neoplasm_status,,
with,20,1019
without,517,97967
